# 03 - S3 object storage with SeaweedFS

Read this notebook and paste each Bash block into an Ubuntu terminal, in order. Keep the same terminal for the commands in this notebook. These are shell instructions; do not run them in a Python kernel.

## Open the project

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



Service: `seaweedfs:8333`. StatefulSet/pod: `seaweedfs` / `seaweedfs-0`. PVC: `seaweedfs-data`, 10 GiB, mounted at `/data`. The manifest pins SeaweedFS 4.47 by digest. The initial `warehouse` bucket supports later exercises. S3 bucket names use hyphens, so this example uses `test-bucket`.

## Apply the S3 manifest

```bash
kubectl apply -f seaweedfs/seaweedfs.yaml
```



## Wait for S3

```bash
kubectl rollout status statefulset/seaweedfs --timeout=600s
```



## Inspect S3 storage

```bash
kubectl get pod seaweedfs-0
kubectl get service seaweedfs
kubectl get pvc seaweedfs-data
kubectl get pv
```

Expect the PVC to be Bound. The PV name is generated; read it from the VOLUME column.

## Forward S3 to this terminal host

```bash
if [[ -f .state/s3-forward.pid ]]; then
  FORWARD_PID=$(cat .state/s3-forward.pid)
  if [[ "$(ps -p "$FORWARD_PID" -o args= 2>/dev/null || true)" == 'kubectl port-forward --address=127.0.0.1 service/seaweedfs 19001:8333' ]]; then kill "$FORWARD_PID"; fi
fi
kubectl port-forward --address=127.0.0.1 service/seaweedfs 19001:8333 > .state/s3-forward.log 2>&1 &
echo $! > .state/s3-forward.pid
```

The process stays in the background. This is host localhost, not a public endpoint.

## Prepare the signed S3 client

```bash
S3_URL=http://127.0.0.1:19001
S3_ACCESS=$(kubectl get secret lab-credentials -o jsonpath='{.data.s3-access-key}' | base64 -d)
S3_SECRET=$(kubectl get secret lab-credentials -o jsonpath='{.data.s3-secret-key}' | base64 -d)
s3() { curl -fsS --aws-sigv4 aws:amz:us-east-1:s3 --user "$S3_ACCESS:$S3_SECRET" "$@"; }
for attempt in $(seq 1 60); do
  if s3 "$S3_URL/warehouse" > .state/warehouse.xml 2>/dev/null; then break; fi
  sleep 1
done
s3 "$S3_URL/warehouse"
```

Expect an S3 XML response for the existing warehouse bucket.

## Create or inspect test-bucket

```bash
BUCKET_STATUS=$(curl -sS -o .state/test-bucket.xml -w '%{http_code}' --aws-sigv4 aws:amz:us-east-1:s3 --user "$S3_ACCESS:$S3_SECRET" "$S3_URL/test-bucket")
case "$BUCKET_STATUS" in
  404) s3 -X PUT "$S3_URL/test-bucket" ;;
  200) cat .state/test-bucket.xml ;;
  *) cat .state/test-bucket.xml; false ;;
esac
```

Creates the bucket if absent and reuses it on repeated runs. A repeated unconditional bucket create would return HTTP 409.


## Write test-object.txt

```bash
printf 'workspace storage example\n' > .state/test-object.txt
OBJECT_HASH=$(sha256sum .state/test-object.txt | cut -d ' ' -f1)
s3 -H "x-amz-content-sha256: $OBJECT_HASH" --upload-file .state/test-object.txt "$S3_URL/test-bucket/test-object.txt"
```



## Read and compare the object

```bash
s3 "$S3_URL/test-bucket/test-object.txt" > .state/test-object-read.txt
cmp .state/test-object.txt .state/test-object-read.txt
cat .state/test-object-read.txt
```

Expect `workspace storage example`; cmp exits successfully.

## List the bucket

```bash
s3 "$S3_URL/test-bucket"
```

Expect `test-object.txt` in the XML response.

## Close the local forward

```bash
kill "$(cat .state/s3-forward.pid)"
rm .state/s3-forward.pid
```



## Check first-start logs and metrics

```bash
python3 validations/check-observability.py --components seaweedfs
```

Monitoring was started in notebook 02. This checks the actual first-start daemon logs and CPU/memory samples without restarting the service to generate logs.


Once Polaris is deployed, stop it before stopping S3: `bash cluster.sh stop s3` handles this dependency. Deleting a pod replaces it using the same PVC; deleting the PVC or Minikube profile destroys the stored objects.

## Stop this service

```bash
kubectl scale statefulset/seaweedfs --replicas=0
kubectl wait --for=delete pod -l app=seaweedfs --timeout=180s
```

This retains its Service and storage. Run after finishing the examples.

## Resume this service

```bash
kubectl scale statefulset/seaweedfs --replicas=1
kubectl rollout status statefulset/seaweedfs --timeout=600s
```

